In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
documents = [
"""
Table: customers

Contains customer information.

Columns:
- customer_id : unique customer identifier
- name : customer name
- email : customer email
- country : customer country
""",

"""
Table: orders

Contains purchase orders.

Columns:
- order_id : unique order identifier
- customer_id : foreign key to customers
- order_date : date of purchase
- amount : order value
""",

"""
Table: products

Contains product catalog.

Columns:
- product_id : unique product identifier
- name : product name
- category : product category
- price : product price
"""
]

In [6]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

vectorstore = Chroma.from_texts(
    texts=documents,
    embedding=OpenAIEmbeddings(
        model="text-embedding-3-small"
    )
)

In [7]:
results = vectorstore.similarity_search(
    "top customers by revenue",
    k=2
)

In [15]:
from langchain.tools import tool

@tool
def search(query: str) -> dict[str, str]:
    """
    Search the database for tables matching the given query.
    """
    return vectorstore.similarity_search(
    query,
    k=2
)

In [18]:
search.invoke('top customers by revenue')

[Document(id='f803b2ce-7e4a-489a-b6e6-3d2c2c39ae2e', metadata={}, page_content='\nTable: customers\n\nContains customer information.\n\nColumns:\n- customer_id : unique customer identifier\n- name : customer name\n- email : customer email\n- country : customer country\n'),
 Document(id='e594f8c9-ad6e-4005-b942-eaa13f68e8ba', metadata={}, page_content='\nTable: orders\n\nContains purchase orders.\n\nColumns:\n- order_id : unique order identifier\n- customer_id : foreign key to customers\n- order_date : date of purchase\n- amount : order value\n')]

In [13]:
results

[Document(id='f803b2ce-7e4a-489a-b6e6-3d2c2c39ae2e', metadata={}, page_content='\nTable: customers\n\nContains customer information.\n\nColumns:\n- customer_id : unique customer identifier\n- name : customer name\n- email : customer email\n- country : customer country\n'),
 Document(id='e594f8c9-ad6e-4005-b942-eaa13f68e8ba', metadata={}, page_content='\nTable: orders\n\nContains purchase orders.\n\nColumns:\n- order_id : unique order identifier\n- customer_id : foreign key to customers\n- order_date : date of purchase\n- amount : order value\n')]

In [19]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name='gpt-5.4-mini',
    temperature=0.0,
)

In [20]:
from langchain.agents import create_agent

agent = create_agent(
    llm,
    tools=[search]
)

In [ ]:
instructions = """
You're a text to sql generator.
You're given a question from a user and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = "What are the top customers by revenue?"

In [ ]:
from langchain_core.messages import HumanMessage

messages = [
        HumanMessage(content=instructions),
        HumanMessage(content=question)
    ]

response = agent.invoke(
    {
        "messages": messages
    }
)